# Latin-Masking Pipeline (Module API)

This notebook runs the latin-masking pipeline step by step using the module-based API.

## Pipeline Steps:
1. **Stage 1** — Normalize → sentence-split → UDPipe → collect adverbs → write `common_adverbs.txt`
2. **Review** — Check `common_adverbs.txt` and `que_blacklist.txt`
3. **Stage 2** — -que split → UDPipe → mask → write masked output

## Step 1: Setup and Imports

In [1]:
from pathlib import Path

from latin_masking import run_pipeline_stage1, run_pipeline_stage2
from latin_masking.types import MaskingConfig

# Configuration
INPUT_DIR = Path("/Users/ben/code/Liber-Regum/Lexical analysis/data")
OUTPUT_DIR = INPUT_DIR
MODEL = "latin-evalatin24-240520"

# Find raw .txt files (excluding already processed files)
txt_files = sorted(
    [
        f
        for f in INPUT_DIR.glob("*.txt")
        if "_sentences" not in f.name
        and "_masked" not in f.name
        and f.name not in ("common_adverbs.txt", "all_adverbs.txt")
    ]
)
print(f"Found {len(txt_files)} files to process:")
for f in txt_files:
    print(f"  - {f.name}")

Found 34 files to process:
  - AE_bhc.txt
  - ASt_troilus.txt
  - BB_adelae.txt
  - BBi_speculum.txt
  - BI_reynardus.txt
  - BM_regum.txt
  - BS_mathematicus.txt
  - EM_mahumeti.txt
  - H?_cilr.txt
  - HA_pentateuchum.txt
  - HM_gestis.txt
  - HW_hortus.txt
  - H_carmina.txt
  - H_mysterio.txt
  - IS_entheticus.txt
  - M_ars.txt
  - M_epistulae.txt
  - M_tobias.txt
  - NLC_miracula.txt
  - NLC_speculum.txt
  - PDE_rebussiculis.txt
  - PR_aurora.txt
  - QS_alexandri.txt
  - RDV_paulino.txt
  - RL_anselmi.txt
  - RT_epistulae.txt
  - RT_memorabilibus.txt
  - RT_miracula.txt
  - SAC_excidio.txt
  - SR_draco.txt
  - U1_pyramo.txt
  - U2_ysengrimus.txt
  - U3_guiardinus.txt
  - X_liberregum.txt


## Step 2: Stage 1 — Normalize, Sentence-Split, UDPipe, Collect Adverbs

Run the first stage of the pipeline. This normalizes text, splits into sentences, processes through UDPipe (with caching), and collects adverbs to build `common_adverbs.txt`.

In [2]:
config = MaskingConfig(
    model=MODEL,
    output_dir=OUTPUT_DIR,
    cache_dir=OUTPUT_DIR / "udpipe_cache",
)

result1 = run_pipeline_stage1(txt_files, config=config)

print(f"Processed {len(result1.sentences_per_file)} files")
print(f"Total sentences: {sum(result1.sentences_per_file.values())}")
print(f"Unique adverbs collected: {len(result1.adverb_counts)}")
print(f"Saved to: {result1.common_adverbs_path}")

# Show top adverbs
from latin_masking.adverbs import normalize_adverb_counts, generate_adverb_list

normalized = normalize_adverb_counts(result1.adverb_counts)
top_adverbs = generate_adverb_list(normalized, 200)
print(f"\nTop 20 adverbs:")
for adv, count in top_adverbs[:20]:
    print(f"  {adv}\t{count}")

  AE_bhc.txt: 399 sentences
    66 adverbs collected
  ASt_troilus.txt: 3495 sentences
    344 adverbs collected
  BB_adelae.txt: 697 sentences
    154 adverbs collected
  BBi_speculum.txt: 981 sentences
    184 adverbs collected
  BI_reynardus.txt: 762 sentences
    178 adverbs collected
  BM_regum.txt: 610 sentences
    100 adverbs collected
  BS_mathematicus.txt: 392 sentences
    108 adverbs collected
  EM_mahumeti.txt: 529 sentences
    164 adverbs collected
  H?_cilr.txt: 789 sentences
    122 adverbs collected
  HA_pentateuchum.txt: 498 sentences
    89 adverbs collected
  HM_gestis.txt: 3471 sentences
    240 adverbs collected
  HW_hortus.txt: 4543 sentences
    236 adverbs collected
  H_carmina.txt: 339 sentences
    95 adverbs collected
  H_mysterio.txt: 310 sentences
    101 adverbs collected
  IS_entheticus.txt: 959 sentences
    144 adverbs collected
  M_ars.txt: 393 sentences
    34 adverbs collected
  M_epistulae.txt: 890 sentences
    69 adverbs collected
  M_tobias.txt

## Step 3: Review Common Adverbs

Before running stage 2, review the generated `common_adverbs.txt` and `que_blacklist.txt`. You can edit these files to add/remove entries as needed.

In [3]:
# Review the generated files
# common_adverbs.txt is written to config.common_adverbs_path (defaults to CWD)
adv_path = result1.common_adverbs_path
print(f"=== common_adverbs.txt ({adv_path}) ===")
if adv_path.exists():
    with open(adv_path) as f:
        for i, line in enumerate(f):
            if i >= 20:
                print("  ...")
                break
            print(f"  {line.strip()}")
else:
    print("  (file not found)")

print("\n=== que_blacklist.txt (first 10 lines) ===")
bl_path = Path("/Users/ben/code/latin-masking/src/latin_masking/data/que_blacklist.txt")
if bl_path.exists():
    with open(bl_path) as f:
        for i, line in enumerate(f):
            if i >= 10:
                print("  ...")
                break
            print(f"  {line.strip()}")
else:
    print("  (file not found)")

=== common_adverbs.txt (common_adverbs.txt) ===
  sic	2607
  iam	1207
  tamen	1036
  inde	1019
  hinc	905
  ergo	864
  tunc	822
  nunc	763
  semper	718
  bene	498
  hic	489
  ibi	479
  simul	464
  magis	447
  nimis	420
  cur	393
  unde	373
  tam	368
  uix	363
  modo	360
  ...

=== que_blacklist.txt (first 10 lines) ===
  absque
  adusque
  antique
  atque
  cuicumque
  cuique
  cuiuscumque
  cuiuscunque
  cuiusque
  cumque
  ...


## Step 4: Stage 2 — -que Split, UDPipe, Mask

Run the second stage. This applies -que splitting using the blacklist + common adverbs, processes through UDPipe (from cache), and applies POS masking.

In [4]:
result2 = run_pipeline_stage2(txt_files, config=config)

print(f"Processed {result2.sentences_processed} sentences")
print(f"Cache hits: {result2.cache_hits}")
print(f"Output files: {[f.name for f in result2.output_files]}")

  AE_bhc.txt: 117 -que splits
    399 masked sentences -> AE_bhc_sentences.quesplit.masked.txt
  ASt_troilus.txt: 625 -que splits
    3495 masked sentences -> ASt_troilus_sentences.quesplit.masked.txt
  BB_adelae.txt: 239 -que splits
    697 masked sentences -> BB_adelae_sentences.quesplit.masked.txt
  BBi_speculum.txt: 306 -que splits
    981 masked sentences -> BBi_speculum_sentences.quesplit.masked.txt
  BI_reynardus.txt: 399 -que splits
    762 masked sentences -> BI_reynardus_sentences.quesplit.masked.txt
  BM_regum.txt: 216 -que splits
    610 masked sentences -> BM_regum_sentences.quesplit.masked.txt
  BS_mathematicus.txt: 224 -que splits
    392 masked sentences -> BS_mathematicus_sentences.quesplit.masked.txt
  EM_mahumeti.txt: 173 -que splits
    529 masked sentences -> EM_mahumeti_sentences.quesplit.masked.txt
  H?_cilr.txt: 251 -que splits
    789 masked sentences -> H?_cilr_sentences.quesplit.masked.txt
  HA_pentateuchum.txt: 251 -que splits
    498 masked sentences -> HA_

## Step 5: View Results

Compare original, sentences, and masked output.

In [5]:
# Show example output
example_file = txt_files[0]
print(f"=== Original ({example_file.name}) ===")
with open(example_file, "r", encoding="utf-8") as f:
    original = f.read()
print(original[:500] + "..." if len(original) > 500 else original)

sentences_file = OUTPUT_DIR / f"{example_file.stem}_sentences.txt"
print(f"\n=== Sentences ({sentences_file.name}) ===")
with open(sentences_file, "r", encoding="utf-8") as f:
    sentences = f.readlines()
for i, sent in enumerate(sentences[:5]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(sentences)} total sentences)")

masked_file = OUTPUT_DIR / f"{example_file.stem}_sentences.quesplit.masked.txt"
print(f"\n=== Masked ({masked_file.name}) ===")
with open(masked_file, "r", encoding="utf-8") as f:
    masked = f.readlines()
for i, sent in enumerate(masked[:5]):
    print(f"{i+1}. {sent.strip()}")
print(f"... ({len(masked)} total sentences)")

=== Original (AE_bhc.txt) ===
Ante dies omnes mundi fuit omnis in uno
	Machina momento facta iubente Deo.
Sed tunc nec celum, nec terra, nec unda, nec aer
	Ornatus habuit quos habet, unde nitet.
Vnda tegit terram, tegit aera, sic elementa
	Hec tria miscentur efficiuntque chaos.
Hec polus empireus superat, ternos ter in ista
	Angelicos cetus collocat arce Deus.
Lumine virtutum cunctis his angelus unus
	Prelucens, dictus Lucifer inde fuit.
Hunc tumor et multos a celo tendit ad ima;
	Qui fuerant humiles promeruere statum.
Nec p...

=== Sentences (AE_bhc_sentences.txt) ===
1. Ante dies omnes mundi fuit omnis in uno <EOL> Machina momento facta iubente Deo.
2. <EOL> Sed tunc nec celum, nec terra, nec unda, nec aer <EOL> Ornatus habuit quos habet, unde nitet.
3. <EOL> Unda tegit terram, tegit aera, sic elementa <EOL> Hec tria miscentur efficiuntque chaos.
4. <EOL> Hec polus empireus superat, ternos ter in ista <EOL> Angelicos cetus collocat arce Deus.
5. <EOL> Lumine uirtutum cunctis his ange